In [6]:
#imports

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import onnxruntime as ort
import matplotlib.gridspec as gridspec

ort.get_device()

'GPU'

In [8]:
# config
MODELS_PATH = Path("../saved_models/cvpr_games_oct_30_onnx/") # relative to ipynb
IMAGES_PATH = Path("../data/objects365_val_patch1/") # relative to ipynb

BASELINE_PATH = Path("baseline_distilled.onnx") # converted to higher opset

# pick a model to measure
some_model = MODELS_PATH/"Sat_Oct_28_18_44_34_2023_starry_night_1_1E05_1E10_1E03.onnx"
torch_model_path = Path("../saved_models/cvpr_games_oct_30/Sat_Oct_28_18_44_34_2023_starry_night_1_1E05_1E10_1E03.pth")

# basic optimisations
OPTIMISED_PATH = Path("optimised.onnx")
FP16_PATH = Path("baseline_fp16.onnx")
IDENTITYNET_PATH = Path("identity.onnx")

# quantization (very slow)
QUANTIZED_PATH = Path("quantized_int8.onnx")

# distilations
RES2_PATH = Path("transformerNetFusion_res2.onnx")
RES1_PATH = Path("transformerNetFusion_res1.onnx")
SEPARATED_PATH = Path("transformerNetFusion_separated.onnx")
SEPARATED_2xSCALED_PATH = Path("transformerNetFusion_separated_2xscaled.onnx")
SEPARATED_4xSCALED_PATH = Path("transformerNetFusion_separated_4xscaled.onnx")

input_shape = (1,3,640,640) # checked beforehand

if not any(MODELS_PATH.glob("*.onnx")):
    raise RuntimeError("Models not found, you didn't clone the GBGST repo or sometthing is very wrong")


PROVIDERS = ["CUDAExecutionProvider"] # dont use tensorrt, or don't try to measure on CPU


# new: baseline image
BASELINE_PNG_PATH = Path("baseline.png")

In [11]:
# get the baseline image
if not BASELINE_PNG_PATH.exists():
    print("Generating baseline image...")
    input_tensor = cv2.imread("example.jpeg")
    input_tensor = cv2.cvtColor(input_tensor, cv2.COLOR_BGR2RGB)
    input_tensor = cv2.resize(input_tensor, (640, 640))
    input_tensor = input_tensor.astype(np.float32)
    input_tensor = np.transpose(input_tensor, (2, 0, 1))
    input_tensor = np.expand_dims(input_tensor, axis=0)

    ort_sess = ort.InferenceSession(str(BASELINE_PATH), providers=PROVIDERS)
    outputs = ort_sess.run(None, {"input": input_tensor})
    
    out_img = np.squeeze(outputs[0], axis=0)
    out_img = np.transpose(out_img, (1, 2, 0))
    out_img = np.clip(out_img, 0, 255).astype(np.uint8)

    cv2.imwrite(str(BASELINE_PNG_PATH), cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR))

In [ ]:
def save_plot_segment(results_subset, baseline_img, filename):
    num_models = len(results_subset)
    fig = plt.figure(figsize=(6, 2.2 * (num_models + 1))) 
    
    gs = gridspec.GridSpec(num_models + 1, 3, 
                           width_ratios=[0.4, 1, 1], 
                           wspace=0.05, 
                           hspace=0.05)

    ax_h_lab = fig.add_subplot(gs[0, 0])
    ax_h_lab.text(0.9, 0.5, "REF", fontweight='bold', ha='right', va='center', transform=ax_h_lab.transAxes)
    ax_h_lab.axis('off')

    ax_h_img = fig.add_subplot(gs[0, 1])
    ax_h_img.imshow(baseline_img)
    ax_h_img.set_aspect('equal', adjustable='box') # Force tight box around image
    ax_h_img.axis('off')

    ax_h_empty = fig.add_subplot(gs[0, 2])
    ax_h_empty.axis('off')

    last_im_delta = None
    for i, res in enumerate(results_subset):
        row = i + 1
        diff = cv2.absdiff(res['image'], baseline_img) * 2
        diff_gray = np.mean(diff, axis=2)

        ax_name = fig.add_subplot(gs[row, 0])
        ax_name.text(0.9, 0.5, res['name'].replace("_","\n"), ha='right', va='center', 
                     fontsize=9, transform=ax_name.transAxes)
        ax_name.axis('off')

        ax_out = fig.add_subplot(gs[row, 1])
        ax_out.imshow(res['image'])
        ax_out.set_aspect('equal', adjustable='box') # Force tight box
        ax_out.axis('off')

        ax_delta = fig.add_subplot(gs[row, 2])
        last_im_delta = ax_delta.imshow(diff_gray, cmap='magma', vmin=0, vmax=100)
        ax_delta.set_aspect('equal', adjustable='box') # Force tight box
        ax_delta.axis('off')

    if last_im_delta:
        cbar_ax = fig.add_axes([0.33, 0.08, 0.6, 0.01]) 
        plt.colorbar(last_im_delta, cax=cbar_ax, orientation='horizontal')
        cbar_ax.tick_params(labelsize=8)

    plt.savefig(filename, dpi=250, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f"Saved: {filename}")


def run_batch_comparison(image_path: Path, baseline_png_path: Path, model_paths: list):
    image_src = cv2.imread(str(image_path))
    image_src = cv2.cvtColor(image_src, cv2.COLOR_BGR2RGB)
    image_src = cv2.resize(image_src, (640, 640))

    if not baseline_png_path.exists():
        raise FileNotFoundError(f"Baseline image not found at {baseline_png_path}")
    
    baseline_img = cv2.imread(str(baseline_png_path))
    baseline_img = cv2.cvtColor(baseline_img, cv2.COLOR_BGR2RGB)
    baseline_img = cv2.resize(baseline_img, (640, 640))

    results = []
    for m_path in model_paths:
        print(f"Inference: {m_path.name}...")
        current_dtype = np.float16 if "16" in m_path.stem else np.float32
        input_tensor = image_src.astype(current_dtype)
        input_tensor = np.transpose(input_tensor, (2, 0, 1))
        input_tensor = np.expand_dims(input_tensor, axis=0)

        ort_sess = ort.InferenceSession(str(m_path), providers=PROVIDERS)
        outputs = ort_sess.run(None, {"input": input_tensor})
        
        out_img = np.squeeze(outputs[0], axis=0)
        out_img = np.transpose(out_img, (1, 2, 0))
        out_img = np.clip(out_img, 0, 255).astype(np.uint8)
        results.append({'name': m_path.stem, 'image': out_img})

    for res in results:
        res['name'] = "_".join(res['name'].split("_")[1:-2])+"_fp16" if "fp" in res['name'] else "_".join(res['name'].split("_")[1:-1])
    
    order = ["res2", "res2_fp16", "res1", "res1_fp16", "separated", "separated_fp16", 
             "separated_2xscaled", "separated_2xscaled_fp16", "separated_4xscaled", "separated_4xscaled_fp16"]
    results.sort(key=lambda x: order.index(x['name']) if x['name'] in order else len(order))

    mid_point = (len(results) + 1) // 2
    save_plot_segment(results[:mid_point], baseline_img, "comparison_part1.png")
    save_plot_segment(results[mid_point:], baseline_img, "comparison_part2.png")

In [15]:
models = list(Path(".").glob("tr*dist*.onnx"))
run_batch_comparison("example.jpeg", BASELINE_PNG_PATH, models)

Inference: transformerNetFusion_res2_distilled_fp16.onnx...


2026-02-12 02:23:45.378365417 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.12: cannot open shared object file: No such file or directory

2026-02-12 02:23:45.378403424 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:1013 CreateExecutionProviderFactoryInstance] Failed to create CUDAExecutionProvider. Require cuDNN 9.* and CUDA 12.*. Please install all dependencies as mentioned in the GPU requirements page (https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements), make sure they're in the PATH, and that your GPU is supported.


Inference: transformerNetFusion_separated_2xscaled_distilled.onnx...


2026-02-12 02:23:45.919338922 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.12: cannot open shared object file: No such file or directory

2026-02-12 02:23:45.919355240 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:1013 CreateExecutionProviderFactoryInstance] Failed to create CUDAExecutionProvider. Require cuDNN 9.* and CUDA 12.*. Please install all dependencies as mentioned in the GPU requirements page (https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements), make sure they're in the PATH, and that your GPU is supported.


Inference: transformerNetFusion_res2_distilled.onnx...


...

Inference: transformerNetFusion_separated_4xscaled_distilled_fp16.onnx...


2026-02-12 02:23:48.444639167 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.12: cannot open shared object file: No such file or directory

2026-02-12 02:23:48.444666691 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:1013 CreateExecutionProviderFactoryInstance] Failed to create CUDAExecutionProvider. Require cuDNN 9.* and CUDA 12.*. Please install all dependencies as mentioned in the GPU requirements page (https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements), make sure they're in the PATH, and that your GPU is supported.
2026-02-12 02:23:48.659186954 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider

Inference: transformerNetFusion_res1_distilled.onnx...
Inference: transformerNetFusion_separated_4xscaled_distilled.onnx...


2026-02-12 02:23:49.167408462 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.12: cannot open shared object file: No such file or directory

2026-02-12 02:23:49.167452145 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:1013 CreateExecutionProviderFactoryInstance] Failed to create CUDAExecutionProvider. Require cuDNN 9.* and CUDA 12.*. Please install all dependencies as mentioned in the GPU requirements page (https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements), make sure they're in the PATH, and that your GPU is supported.


Saved: comparison_part1.png
Saved: comparison_part2.png
